In [2]:
11

11

In [3]:
import os
import random
from tqdm import tqdm
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Any
import sentencepiece as spm

import evaluate
import numpy as np
import torch
from datasets import load_dataset, DatasetDict, Dataset, concatenate_datasets, disable_progress_bar
from transformers import (
    pipeline,
    MBart50TokenizerFast,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    set_seed
)
from transformers.trainer_utils import get_last_checkpoint

2025-11-02 05:13:50.405735: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-02 05:13:50.490116: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-02 05:13:51.740669: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [4]:
# Replace with your HuggingFace dataset name
DATASET_NAME = "pollitoconpapass/spanish_2_shipibo_konibo"
# Adjust these if your dataset has different column names
SOURCE_COLUMN = "shp"  # or "shipibo"
TARGET_COLUMN = "spa"   # or "spanish"


SPA_MODEL_NAME = "pysentimiento/robertuito-sentiment-analysis"
SPA_MODEL_LOCAL_PATH = "models/pysentimiento/robertuito-sentiment-analysis"
SPA_TOKENIZER_NAME = "pysentimiento/robertuito-sentiment-analysis"
SPA_TOKENIZER_LOCAL_PATH = "models/pysentimiento/robertuito-sentiment-analysis"


BASE_MODEL_NAME = "xlm-roberta-base"
BASE_MODEL_LOCAL_PATH = "models/xlm-roberta-base"
BASE_TOKENIZER_NAME = "xlm-roberta-base"
BASE_TOKENIZER_LOCAL_PATH = "tokenizers/xlm-roberta-base"  # Assuming tokenizer is saved with the model

TRAINED_MODEL_NAME = "bert-sentiment-shipibo"
TRAINED_MODEL_OUTPUT_DIR = f"models/bert-sentiment-shipibo"
TRAINED_MODEL_LOGS_DIR = f"logs/bert-sentiment-shipibo"

CONFIDENCE_THRESHOLD = 0.6

In [5]:
def clear_gpu_memory():
    """Clear GPU memory cache"""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        import gc
        gc.collect()
        print("🧹 GPU memory cleared")

In [6]:
print(f"📊 Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

📊 Device: GPU
   GPU: NVIDIA GeForce RTX 4070 Ti SUPER
   Memory: 17.17 GB


In [7]:
def load_shipibo_dataset(dataset_name: str) -> DatasetDict:
    """
    Load Shipibo-Spanish parallel corpus from HuggingFace
    
    Args:
        dataset_name: HuggingFace dataset identifier (e.g., "username/dataset-name")
    
    Returns:
        DatasetDict with train/validation/test splits
    """
    print(f"\n📥 Loading dataset: {dataset_name}")
    
    # Load the dataset
    dataset = load_dataset(dataset_name)
    
    # Display statistics
    print("\n📊 Dataset Statistics:")
    for split in dataset.keys():
        print(f"   {split}: {len(dataset[split])} examples")
        if len(dataset[split]) > 0:
            # Show first example
            example = dataset[split][0]
            print(f"   Example Shipibo: {example.get('shp', example.get('shipibo', 'N/A'))[:50]}...")
            print(f"   Example Spanish: {example.get('spa', example.get('spanish', 'N/A'))[:50]}...")
    
    return dataset


In [8]:
def prepare_tokenizer(tokenizer_path: str, source_key: str, target_key: str) -> AutoTokenizer:
    """
    Load and configure mBART-50 tokenizer for Shipibo→Spanish
    """
    print("\n🔧 Loading mBART-50 tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
        
    return tokenizer

In [37]:
def prepare_sentiment_model(model_name: str, model_path: str, num_labels: int):
    """
    Get the pre-trained model locally.
    
    Args:
        model_name: Name of the pre-trained model to download.
        model_path: Directory where the model is saved.
    """
    print(f"\n⬇️  Loading model: {model_name}")
    model = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels=num_labels, ignore_mismatched_sizes=True)
    model.config.label2id = {'NEG': 0, 'NEU': 1, 'POS': 2}
    model.config.id2label = {0: 'NEG', 1: 'NEU', 2: 'POS'}
    print(f"   Model loaded from: {model_path}")
    return model
    

In [10]:
def download_sentiment_model(model_name: str, output_dir: str):
    """
    Download and save the pre-trained model locally.
    
    Args:
        model_name: Name of the pre-trained model to download.
        output_dir: Directory to save the downloaded model.
    """
    print(f"\n⬇️  Downloading model: {model_name}")
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    model.save_pretrained(output_dir)
    print(f"   Model saved to: {output_dir}")

In [11]:
def download_tokenizer(tokenizer_name: str, output_dir: str):
    """
    Download and save the tokenizer locally.
    
    Args:
        tokenizer_name: Name of the tokenizer to download.
        output_dir: Directory to save the downloaded tokenizer.
    """
    print(f"\n⬇️  Downloading tokenizer: {tokenizer_name}")
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    tokenizer.save_pretrained(output_dir)
    print(f"   Tokenizer saved to: {output_dir}")

In [12]:

def project_labels(parallel_corpus: Dataset, model: AutoModelForSequenceClassification, tokenizer: AutoTokenizer, threshold: float) -> Dataset:
    """
    Project sentiment labels from Spanish to Shipibo using a Spanish sentiment model to predict sentiment scores and then project them into the parallel corpus.

    Args:
        parallel_corpus: The parallel corpus containing Spanish and Shipibo text.
        model: The Spanish sentiment model used for prediction.
        tokenizer: The tokenizer corresponding to the sentiment model.
    Returns:
        A dataset containing sentiment labels with alligned parallel sentences. (shipibo, spanish, sentiment_label)
    """

    def predict_labels(texts: List[str]) -> List[Dict]:
        """Predict sentiment label for given Spanish text. Returns both label and score."""
        pipe = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)
        result = pipe(texts)
        return result

    # Predict labels for each Spanish sentence in the parallel corpus
    predicted_labels = []
    predicted_scores = []
    
    # Batch process Spanish texts
    for i in tqdm(range(0, len(parallel_corpus), 1000)):
        batch = parallel_corpus[i:i+1000]['spa']
        results = predict_labels(batch)
        for item in results:
            predicted_labels.append(item['label'])
            predicted_scores.append(item['score'])
            
    parallel_corpus = parallel_corpus.add_column("sentiment_label", predicted_labels)
    parallel_corpus = parallel_corpus.add_column("sentiment_score", predicted_scores)

    # Apply confidence threshold
    parallel_corpus = parallel_corpus.filter(lambda x: x['sentiment_score'] >= threshold)
    return parallel_corpus

## Project Labels

In [13]:
# Download dataset
dataset = load_shipibo_dataset(DATASET_NAME)


📥 Loading dataset: pollitoconpapass/spanish_2_shipibo_konibo

📊 Dataset Statistics:
   train: 24144 examples
   Example Shipibo: Nokawe....
   Example Spanish: Apágalo....
   validation: 3018 examples
   Example Shipibo: píarebo....
   Example Spanish: punta de flecha....
   test: 3018 examples
   Example Shipibo: Kirikanin non raoki ika rabiti wishaxon axetixobon...
   Example Spanish: Escribe en un cuaderno un poema a nuestras plantas...


In [14]:
# Download Spanish model and tokenizer if not already present
if not os.path.exists(SPA_MODEL_LOCAL_PATH):
    download_sentiment_model(SPA_MODEL_NAME, SPA_MODEL_LOCAL_PATH)
spa_model = prepare_sentiment_model(SPA_MODEL_NAME, SPA_MODEL_LOCAL_PATH)
if not os.path.exists(SPA_TOKENIZER_LOCAL_PATH):
    download_tokenizer(SPA_TOKENIZER_NAME, SPA_TOKENIZER_LOCAL_PATH)
spa_tokenizer = prepare_tokenizer(SPA_TOKENIZER_NAME, SOURCE_COLUMN, TARGET_COLUMN)


⬇️  Loading model: pysentimiento/robertuito-sentiment-analysis
   Model loaded from: models/pysentimiento/robertuito-sentiment-analysis

🔧 Loading mBART-50 tokenizer...


## Fine-Tune Sentiment Model

In [38]:
# Download Base model and tokenizer if not already present
if not os.path.exists(BASE_MODEL_LOCAL_PATH):
    download_sentiment_model(BASE_MODEL_NAME, BASE_MODEL_LOCAL_PATH)
base_model = prepare_sentiment_model(BASE_MODEL_NAME, BASE_MODEL_LOCAL_PATH, 3)
if not os.path.exists(BASE_TOKENIZER_LOCAL_PATH):
    download_tokenizer(BASE_TOKENIZER_NAME, BASE_TOKENIZER_LOCAL_PATH)
base_tokenizer = prepare_tokenizer(BASE_TOKENIZER_NAME, SOURCE_COLUMN, TARGET_COLUMN)

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at models/xlm-roberta-base and are newly initialized because the shapes did not match:
- classifier.out_proj.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([3]) in the model instantiated
- classifier.out_proj.weight: found shape torch.Size([2, 768]) in the checkpoint and torch.Size([3, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



⬇️  Loading model: xlm-roberta-base
   Model loaded from: models/xlm-roberta-base

🔧 Loading mBART-50 tokenizer...


In [18]:
# Get labels for test and validation sets
projected_train = project_labels(dataset['train'], spa_model, spa_tokenizer, CONFIDENCE_THRESHOLD)
projected_test = project_labels(dataset['test'], spa_model, spa_tokenizer, CONFIDENCE_THRESHOLD)
projected_validation = project_labels(dataset['validation'], spa_model, spa_tokenizer, CONFIDENCE_THRESHOLD)

100%|██████████| 4/4 [00:19<00:00,  4.94s/it]


In [19]:
class ConfidenceWeightedTrainer(Trainer):
    """
    Custom trainer that weights loss by label confidence.
    Includes sanity checks to catch sneaky CUDA gremlins before they explode!
    """

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # --- Extract label and confidence (weight) ---
        try:
            weights = inputs.pop("weight")  # Confidence scores
            labels = inputs.pop("labels")    # True labels
        except KeyError as e:
            raise KeyError(f"Missing expected key in inputs: {e}. "
                           f"Available keys: {list(inputs.keys())}")

        # --- Type and shape sanity checks ---
        if not torch.is_tensor(weights):
            weights = torch.tensor(weights, dtype=torch.float32, device=model.device)
        if not torch.is_tensor(labels):
            labels = torch.tensor(labels, dtype=torch.long, device=model.device)

        # Convert datatypes if needed
        if labels.dtype != torch.long:
            print("⚠️ Converting labels to torch.long")
            labels = labels.long()
        if weights.dtype != torch.float:
            print("⚠️ Converting weights to torch.float")
            weights = weights.float()

        # --- Forward pass ---
        outputs = model(**inputs)
        logits = outputs.logits

        # --- Defensive checks against CUDA betrayal ---
        if torch.isnan(logits).any():
            raise ValueError("🚨 Logits contain NaN values!")

        if logits.shape[-1] <= labels.max():
            raise ValueError(
                f"🚨 Logits last dim ({logits.shape[-1]}) is <= max label ({labels.max().item()})!"
            )

        if labels.min() < 0:
            raise ValueError(f"🚨 Negative label found: {labels.min().item()}")

        # --- Weighted Cross Entropy Loss ---
        loss_fct = torch.nn.CrossEntropyLoss(reduction='none')
        per_sample_loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))

        # Broadcast weights to same shape if needed
        if weights.ndim == 1 and weights.shape[0] != per_sample_loss.shape[0]:
            weights = weights.view(-1)
            if weights.shape[0] != per_sample_loss.shape[0]:
                raise ValueError(f"🚨 Weight size {weights.shape} != loss size {per_sample_loss.shape}")

        weighted_loss = (per_sample_loss * weights).mean()

        # --- Optional dramatic confirmation ---
        if torch.isnan(weighted_loss):
            raise ValueError("💥 Weighted loss is NaN! The evil has escaped containment!")

        return (weighted_loss, outputs) if return_outputs else weighted_loss


In [20]:
@dataclass
class DataCollatorWithWeights:
    """
    Custom data collator that preserves the 'weight' field.
    """
    tokenizer: Any
    padding: bool = True
    max_length: int = None
    pad_to_multiple_of: int = None
    
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
                
        # Extract weights before collating
        weights = torch.tensor([f['weight'] for f in features], dtype=torch.float)
        labels = torch.tensor([f['label'] for f in features], dtype=torch.long)
        
        # Remove weight and label from features for tokenizer collation
        batch = {
            'input_ids': [f['input_ids'] for f in features],
            'attention_mask': [f['attention_mask'] for f in features],
        }
        
        # Pad sequences
        batch = self.tokenizer.pad(
            batch,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors='pt'
        )
        
        # Add back weights and labels
        batch['weight'] = weights
        batch['labels'] = labels  # Use 'labels' not 'label'
        
        return batch

In [21]:
def preprocess_function(dataset: Dataset, tokenizer):
    """Tokenize and encode the Shipibo texts."""
    encodings = tokenizer(dataset["shp"], truncation=True, padding='max_length', max_length=128)
    # Map sentiment labels to IDs
    label_map = {'NEG': 0, 'NEU': 1, 'POS': 2}
    encodings['sentiment_label'] = [label_map[label] for label in dataset['sentiment_label']]
    return encodings

In [22]:
# Preprocess datasets
train_dataset = projected_train.map(preprocess_function, fn_kwargs={"tokenizer": base_tokenizer}, batched=True)
test_dataset = projected_test.map(preprocess_function, fn_kwargs={"tokenizer": base_tokenizer}, batched=True)
validation_dataset = projected_validation.map(preprocess_function, fn_kwargs={"tokenizer": base_tokenizer}, batched=True)


# Format datasets for PyTorch
train_dataset = train_dataset.rename_column("sentiment_label", "label")
train_dataset = train_dataset.rename_column("sentiment_score", "weight")
train_dataset = train_dataset.remove_columns(["shp", "spa", "__index_level_0__"])

test_dataset = test_dataset.rename_column("sentiment_label", "label")
test_dataset = test_dataset.rename_column("sentiment_score", "weight")
test_dataset = test_dataset.remove_columns(["shp", "spa", "__index_level_0__"])

validation_dataset = validation_dataset.rename_column("sentiment_label", "label")
validation_dataset = validation_dataset.rename_column("sentiment_score", "weight")
validation_dataset = validation_dataset.remove_columns(["shp", "spa", "__index_level_0__"])

Map:   0%|          | 0/2075 [00:00<?, ? examples/s]

In [23]:
# Training hyperparameters
BATCH_SIZE = 4           # Adjust based on GPU memory
LEARNING_RATE = 3e-5
NUM_EPOCHS = 10
WARMUP_STEPS = 500
SEED = 42

In [24]:
# Training arguments
training_args = TrainingArguments(
    output_dir=TRAINED_MODEL_OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    warmup_steps=WARMUP_STEPS,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    warmup_ratio=0.1,
    logging_steps=50,
    logging_dir=TRAINED_MODEL_LOGS_DIR,
    report_to="none",  # Disable wandb/tensorboard
    fp16=torch.cuda.is_available(),  # Use mixed precision if GPU available,
    remove_unused_columns=False,  # Important for custom collator
)

In [41]:
data_collator = DataCollatorWithWeights(
    tokenizer=base_tokenizer,
    padding=True,
    max_length=128
)

trainer = ConfidenceWeightedTrainer(
    model=base_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    tokenizer=base_tokenizer,
    data_collator=data_collator,
)

/tmp/ipykernel_825503/803511522.py:7: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `ConfidenceWeightedTrainer.__init__`. Use `processing_class` instead.
  trainer = ConfidenceWeightedTrainer(


In [ ]:
# Train
print(f"\n{'='*60}")
print("Starting training...")
print(f"{'='*60}\n")

trainer.train()

# Save final model
print(f"\nSaving model to {TRAINED_MODEL_OUTPUT_DIR}...")
trainer.save_model(TRAINED_MODEL_OUTPUT_DIR)
trainer.tokenizer.save_pretrained(TRAINED_MODEL_OUTPUT_DIR)
print("\n✓ Training complete!")
clear_gpu_memory()

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.



Starting training...



Epoch,Training Loss,Validation Loss
1,0.608400,0.590907
2,0.540500,0.589140
3,0.407900,0.634859
4,0.547700,0.522694
5,0.508000,0.554306
6,0.402000,0.553942
7,0.518500,0.517690
8,0.476900,0.529109
9,0.418400,0.506712
10,0.436000,0.507844


/home/chech/torch/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:2779: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
/home/chech/torch/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:2779: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
/home/chech/torch/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:2779: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
/home/chech/torch/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:2779: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_len


Saving model to models/bert-sentiment-shipibo...


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.



✓ Training complete!


In [28]:
import torch

# Check labels are in valid range
print("Checking labels...")
all_labels = [item['label'] for item in train_dataset]
print(f"Min label: {min(all_labels)}")
print(f"Max label: {max(all_labels)}")
print(f"Unique labels: {set(all_labels)}")
print(f"Model expects labels in range: [0, {base_model.config.num_labels - 1}]")

# Check for NaN/Inf in weights
print("\nChecking weights...")
all_weights = [item['weight'] for item in train_dataset]
print(f"Min weight: {min(all_weights)}")
print(f"Max weight: {max(all_weights)}")
print(f"Any NaN: {any(w != w for w in all_weights)}")  # NaN != NaN is True
print(f"Any Inf: {any(w == float('inf') or w == float('-inf') for w in all_weights)}")

# Check a sample batch
print("\nChecking sample batch...")
sample_batch = data_collator([train_dataset[i] for i in range(4)])
print(f"Labels shape: {sample_batch['labels'].shape}")
print(f"Labels: {sample_batch['labels']}")
print(f"Weights shape: {sample_batch['weight'].shape}")
print(f"Weights: {sample_batch['weight']}")
print(f"Input_ids shape: {sample_batch['input_ids'].shape}")

Checking labels...
Min label: 0
Max label: 2
Unique labels: {0, 1, 2}
Model expects labels in range: [0, 2]

Checking weights...
Min weight: 0.6000428795814514
Max weight: 0.9854215383529663
Any NaN: False
Any Inf: False

Checking sample batch...
Labels shape: torch.Size([4])
Labels: tensor([1, 1, 1, 1])
Weights shape: torch.Size([4])
Weights: tensor([0.7868, 0.6026, 0.6533, 0.6892])
Input_ids shape: torch.Size([4, 128])


In [43]:
clear_gpu_memory()

🧹 GPU memory cleared
